# 02 (v2) — Temperatura de superfície (LST) e NDVI — Landsat 8/9 no Google Earth Engine
O UrbVerde só exporta PNG; aqui reproduzimos o dado de origem (Landsat C2 L2, banda `ST_B10`).

Gera:
1. **Composto multibanda (GeoTIFF 30 m, EPSG:31983)**: LST mín/média/mediana/p90/máx/desvio, anomalia média/p90, NDVI mediano
   → usado no notebook 03 (estatística zonal ponderada por fração de pixel).
2. **Série temporal por escola** (CSV): LST e anomalia médias no buffer de 100 m, cena a cena
   → variação temporal (mín/máx/amplitude entre datas).

**Versão 2 do pipeline:** o processamento roda nos servidores do Google e só os resultados vêm para o Drive (sem baixar cena por cena). Saída em `data/raw/lst_gee/`; a v1 (`02b`, em `data/raw/lst/`) fica intacta. A última célula compara as duas.

Pré-requisito: projeto Google Cloud com Earth Engine habilitado (gratuito para uso não comercial/pesquisa).

In [ ]:
!pip -q install earthengine-api geemap geopandas

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import ee, geemap, geopandas as gpd, pandas as pd, time, shutil, glob
GEE_PROJECT = 'SEU-PROJETO-GCP'   # <<< ID do seu projeto Google Cloud registrado no Earth Engine
ee.Authenticate(); ee.Initialize(project=GEE_PROJECT)

INI, FIM = '2019-01-01', '2026-01-01'
MAX_NUVEM = 40           # % de nuvem da cena
MIN_VALIDO = 0.6         # fração mínima de pixels válidos na área de estudo
PASTA_GEE = 'GEE_ESCOLAS_LST'   # pasta criada na raiz do Meu Drive pelo export

In [ ]:
esc = gpd.read_file(f'{INT}/escolas.gpkg', layer='escolas')
buf = gpd.read_file(f'{INT}/escolas.gpkg', layer=f'buffer_{BUFFER_REF}m')
aoi_gdf = gpd.GeoDataFrame(geometry=[esc.unary_union.convex_hull.buffer(3000)], crs=CRS).to_crs(4326)
aoi = geemap.gdf_to_ee(aoi_gdf).geometry()
fc_buf = geemap.gdf_to_ee(buf.to_crs(4326)[['cod_esc', 'geometry']])

In [ ]:
def prep(img):
    qa = img.select('QA_PIXEL')
    limpo = (qa.bitwiseAnd(1 << 1).eq(0)        # nuvem dilatada
             .And(qa.bitwiseAnd(1 << 3).eq(0))  # nuvem
             .And(qa.bitwiseAnd(1 << 4).eq(0))  # sombra
             .And(qa.bitwiseAnd(1 << 5).eq(0))) # neve
    lst = img.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15).rename('LST')
    sr = img.select(['SR_B4', 'SR_B5']).multiply(0.0000275).add(-0.2)
    ndvi = sr.normalizedDifference(['SR_B5', 'SR_B4']).rename('NDVI')
    return lst.addBands(ndvi).updateMask(limpo).copyProperties(img, ['system:time_start', 'CLOUD_COVER'])

n_total = ee.Number(ee.Image.constant(1).clip(aoi).reduceRegion(ee.Reducer.count(), aoi, 90, maxPixels=1e10).get('constant'))

def add_anom(img):
    st = img.select('LST').reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), '', True), aoi, 90, maxPixels=1e10)
    m = ee.Number(st.get('LST_mean'))
    return (img.addBands(img.select('LST').subtract(m).rename('ANOM'))
               .set({'lst_media_area': m, 'frac_valida': ee.Number(st.get('LST_count')).divide(n_total)}))

meses_letivos_quentes = ee.Filter.Or(ee.Filter.calendarRange(2, 4, 'month'), ee.Filter.calendarRange(10, 12, 'month'))
col = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').merge(ee.ImageCollection('LANDSAT/LC09/C02/T1_L2'))
       .filterBounds(aoi).filterDate(INI, FIM).filter(meses_letivos_quentes)
       .filter(ee.Filter.lt('CLOUD_COVER', MAX_NUVEM))
       .map(prep).map(add_anom).filter(ee.Filter.gte('frac_valida', MIN_VALIDO)))
print('cenas usadas:', col.size().getInfo())

In [ ]:
red = (ee.Reducer.min().combine(ee.Reducer.mean(), '', True).combine(ee.Reducer.median(), '', True)
       .combine(ee.Reducer.percentile([90]), '', True).combine(ee.Reducer.max(), '', True)
       .combine(ee.Reducer.stdDev(), '', True))
comp = (col.select('LST').reduce(red)
        .addBands(col.select('ANOM').reduce(ee.Reducer.mean().combine(ee.Reducer.percentile([90]), '', True)))
        .addBands(col.select('NDVI').median().rename('NDVI_median'))
        .toFloat())
print(comp.bandNames().getInfo())

t1 = ee.batch.Export.image.toDrive(image=comp, description='lst_composto', folder=PASTA_GEE,
                                   fileNamePrefix='lst_composto', region=aoi, scale=30, crs=CRS, maxPixels=1e10)
t1.start()

In [ ]:
# série temporal: média ponderada por fração de pixel (o reducer mean do GEE pondera pela cobertura do polígono)
def por_cena(img):
    d = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    return (img.select(['LST', 'ANOM']).reduceRegions(fc_buf, ee.Reducer.mean(), scale=30, tileScale=4)
               .map(lambda f: f.set('data', d)))
serie = col.map(por_cena).flatten()
t2 = ee.batch.Export.table.toDrive(collection=serie, description='lst_serie_escolas', folder=PASTA_GEE,
                                   fileNamePrefix='lst_serie_escolas', fileFormat='CSV',
                                   selectors=['cod_esc', 'data', 'LST', 'ANOM'])
t2.start()

In [ ]:
# aguarda os exports e copia para data/raw/lst
while any(t.status()['state'] in ('READY', 'RUNNING') for t in [t1, t2]):
    print([t.status()['state'] for t in [t1, t2]]); time.sleep(60)
print([t.status() for t in [t1, t2]])
DEST = f'{RAW}/lst_gee'   # v2: pasta separada, não sobrescreve a v1 (Planetary Computer)
os.makedirs(DEST, exist_ok=True)
for f in glob.glob(f'/content/drive/MyDrive/{PASTA_GEE}/*'):
    shutil.copy(f, DEST + '/'); print('copiado', os.path.basename(f))

In [ ]:
# comparação v1 (Planetary Computer) × v2 (Earth Engine) — mesma escola, mesmo raio de 100 m
v1 = glob.glob(f'{RAW}/lst/lst_serie_escolas*.csv'); v2 = glob.glob(f'{DEST}/lst_serie_escolas*.csv')
if v1 and v2:
    a = pd.read_csv(v1[0], dtype={'cod_esc': str}).groupby('cod_esc')[['LST', 'ANOM']].mean()
    b = pd.read_csv(v2[0], dtype={'cod_esc': str}).groupby('cod_esc')[['LST', 'ANOM']].mean()
    m = a.join(b, lsuffix='_v1', rsuffix='_v2').dropna()
    print('escolas comparadas:', len(m))
    for c in ['LST', 'ANOM']:
        print(c, '| Pearson', round(m[f'{c}_v1'].corr(m[f'{c}_v2']), 3),
              '| Spearman (ordem)', round(m[f'{c}_v1'].corr(m[f'{c}_v2'], method='spearman'), 3),
              '| dif. média °C', round((m[f'{c}_v2'] - m[f'{c}_v1']).mean(), 2))
    print('Critério: Spearman da ANOM ≥ 0,95 → v2 pode substituir a v1 (copiar lst_gee/* para lst/).')
else:
    print('Falta uma das versões para comparar.')